# Stage 2.4 - Foundation evidence aggregation

This notebook validates only the Stage 2 decoder-foundation schema. Full five-fold decoder aggregation, statistical testing, and thesis comparisons remain blocked until Stage 3.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

STAGE2_SCHEMA="foundation_stage2_v1"
BONES=["femur","tibia","patella","fibula"]
ARMS={"plain_unet_style","residual_vnet_style"}
SUBSETS={"subset_1_healthy","subset_2_mixed"}
RUN_AGGREGATION=False

def find_project_root(start):
    for candidate in [Path(start).resolve(),*Path(start).resolve().parents]:
        if (candidate/"configs"/"baseline_protocol_v1.json").exists(): return candidate
    raise FileNotFoundError("project root not found")
ROOT=find_project_root(Path.cwd())
SUMMARY_JSON=ROOT/"models"/"decoders"/STAGE2_SCHEMA/"fold_0"/"foundation_summary.json"

In [ ]:
from scipy.ndimage import binary_erosion, distance_transform_edt, label

BONES = ["femur", "tibia", "patella", "fibula"]


def overlap_metrics(prediction, target):
    """Explicit empty handling: an empty target is invalid; an empty prediction against a target scores zero."""
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    invalid_target = not target.any(); empty_prediction = not prediction.any()
    if invalid_target:
        return {"dice": float("nan"), "iou": float("nan"), "invalid_target": True, "empty_prediction": empty_prediction}
    if empty_prediction:
        return {"dice": 0.0, "iou": 0.0, "invalid_target": False, "empty_prediction": True}
    intersection = np.logical_and(prediction, target).sum(dtype=np.float64)
    pred_count = prediction.sum(dtype=np.float64); target_count = target.sum(dtype=np.float64)
    return {"dice": float(2 * intersection / (pred_count + target_count)), "iou": float(intersection / (pred_count + target_count - intersection)), "invalid_target": False, "empty_prediction": False}


def _surface_distances(prediction, target, spacing_xyz):
    prediction = np.asarray(prediction, dtype=bool); target = np.asarray(target, dtype=bool)
    spacing_xyz = tuple(float(value) for value in spacing_xyz)
    if len(spacing_xyz) != 3 or any(value <= 0 for value in spacing_xyz): raise ValueError("spacing_xyz must contain three positive millimetre values")
    pred_surface = prediction & ~binary_erosion(prediction); target_surface = target & ~binary_erosion(target)
    if not pred_surface.any() or not target_surface.any(): return None
    to_target = distance_transform_edt(~target_surface, sampling=spacing_xyz)[pred_surface]
    to_prediction = distance_transform_edt(~pred_surface, sampling=spacing_xyz)[target_surface]
    return np.concatenate([to_target, to_prediction])


def hd95_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(np.percentile(distances, 95))


def assd_mm(prediction, target, spacing_xyz):
    distances = _surface_distances(prediction, target, spacing_xyz)
    return float("nan") if distances is None else float(distances.mean())


def per_bone_metrics(logits, target, spacing_xyz=(0.78125, 0.78125, 0.78125), threshold=0.5):
    probability = torch.sigmoid(logits.float()).detach().cpu().numpy(); truth = target.detach().cpu().numpy() > 0.5
    prediction = probability > threshold; rows = []
    for batch_index in range(prediction.shape[0]):
        record = {}
        for bone_index, bone in enumerate(BONES):
            pred_mask = prediction[batch_index, bone_index]; target_mask = truth[batch_index, bone_index]
            overlap = overlap_metrics(pred_mask, target_mask)
            record.update({f"dice_{bone}": overlap["dice"], f"iou_{bone}": overlap["iou"], f"hd95_mm_{bone}": hd95_mm(pred_mask, target_mask, spacing_xyz), f"assd_mm_{bone}": assd_mm(pred_mask, target_mask, spacing_xyz), f"empty_prediction_{bone}": overlap["empty_prediction"], f"invalid_target_{bone}": overlap["invalid_target"]})
        valid_dice = [record[f"dice_{bone}"] for bone in BONES if np.isfinite(record[f"dice_{bone}"])]
        valid_iou = [record[f"iou_{bone}"] for bone in BONES if np.isfinite(record[f"iou_{bone}"])]
        record["dice_macro"] = float(np.mean(valid_dice)) if valid_dice else float("nan"); record["iou_macro"] = float(np.mean(valid_iou)) if valid_iou else float("nan")
        rows.append(record)
    return rows


def two_largest_components(mask):
    labels, count = label(np.asarray(mask, dtype=bool))
    if count < 2: return None
    sizes = [(component, int((labels == component).sum())) for component in range(1, count + 1)]
    selected = sorted(sizes, key=lambda item: item[1], reverse=True)[:2]
    return labels == selected[0][0], labels == selected[1][0]


def minimum_component_gap_mm(mask, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    pair = two_largest_components(mask)
    if pair is None: return float("nan")
    first, second = pair
    return float(distance_transform_edt(~second, sampling=spacing_xyz)[first].min())


def component_bridge_metrics(prediction, target, spacing_xyz=(0.78125, 0.78125, 0.78125)):
    _, pred_components = label(np.asarray(prediction, dtype=bool)); _, target_components = label(np.asarray(target, dtype=bool))
    return {"prediction_components": int(pred_components), "target_components": int(target_components), "component_agreement": bool(pred_components == target_components), "false_bridge": bool(target_components >= 2 and pred_components < target_components), "prediction_min_gap_mm": minimum_component_gap_mm(prediction, spacing_xyz), "target_min_gap_mm": minimum_component_gap_mm(target, spacing_xyz)}


def aggregate_subject_level(frame, metric_columns, subject_column="subject_id"):
    """Average knees within subject first, then average subjects so bilateral knees do not receive extra weight."""
    subject = frame.groupby(subject_column, as_index=False)[metric_columns].mean(numeric_only=True)
    return subject, subject[metric_columns].mean(numeric_only=True).to_dict()

In [ ]:
def validate_foundation_summary(path=SUMMARY_JSON):
    if not path.is_file(): raise FileNotFoundError(path)
    summary=json.loads(path.read_text(encoding="utf-8"))
    if summary.get("schema_version")!=STAGE2_SCHEMA or summary.get("fold")!=0: raise RuntimeError("summary schema/fold mismatch")
    rows=summary.get("results",[])
    observed={(row["arm"],row["subset"]) for row in rows}; expected={(arm,subset) for arm in ARMS for subset in SUBSETS}
    if observed!=expected: raise RuntimeError(f"missing/extra foundation runs: {observed ^ expected}")
    if len({row["feature_cache_sha256"] for row in rows})!=1: raise RuntimeError("decoder arms did not use one identical feature cache")
    for row in rows:
        if not row.get("pass") or not row.get("resume_pass"): raise RuntimeError(f"failed gate row: {row['arm']}/{row['subset']}")
        if not row.get("resume_parameter_sha256") or not row.get("resume_logits_sha256") or not row.get("config_sha256"): raise RuntimeError(f"missing resume/config hashes: {row['arm']}/{row['subset']}")
        if row.get("peak_host_bytes") is None: raise RuntimeError(f"missing peak host-memory evidence: {row['arm']}/{row['subset']}")
        if any(float(row[f"minimum_dice_{bone}"])<0.90 for bone in BONES): raise RuntimeError(f"per-case/per-bone Dice gate failed: {row['arm']}/{row['subset']}")
        if row.get("gpu_headroom_fraction") is None or float(row["gpu_headroom_fraction"])<0.10: raise RuntimeError(f"resource headroom gate failed: {row['arm']}/{row['subset']}")
    return pd.DataFrame(rows),summary

if RUN_AGGREGATION:
    frame,summary=validate_foundation_summary(); display(frame); print("Stage 2 decoder foundation PASS",summary["feature_cache_sha256"])
else:
    print("Aggregation disabled until the returned HPC decoder bundle is independently reviewed.")